# Performance Analysis & Research Report: CSIRO Biomass Regression

## 1. Introduction and Objectives
This report provides a comprehensive analysis of the deep learning models developed for the CSIRO Image2Biomass competition. The objective is to estimate 5 biomass targets (e.g., `Dry_Total_g`, `GDM_g`) from pasture images.

We compare the efficacy of two primary architectures, **ResNet18** and **DenseNet121**, and evaluate the impact of advanced training strategies including **Test Time Augmentation (TTA)**, **MixUp**, and **Weighted Huber Loss**.

### Key Findings Summary
- **Best Public Score**: **0.52 R2** (ResNet18 Ensemble, v2)
- **Best Private Score**: **0.48 R2** (ResNet18 Ensemble, v2)
- **Insight**: The simpler ResNet18 architecture generalized better than the deeper DenseNet121, potentially due to the small size of the dataset and the textural nature of the features.

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Setting a professional style
sns.set_theme(style="whitegrid", context="talk")

# Aggregated Experiment Data
data = {
    "Model": [
        "ResNet18 Sub2", "ResNet18 Sub3", "ResNet18 Sub1",
        "DenseNet121 v25", "DenseNet121 v21", "DenseNet121 v18", "DenseNet121 v15", "Hybrid ViT v3"
    ],
    "Architecture": ["ResNet18", "ResNet18", "ResNet18", "DenseNet121", "DenseNet121", "DenseNet121", "DenseNet121", "ViT"],
    "Private Score": [0.48, 0.42, 0.39, 0.33, 0.28, 0.30, 0.17, 0.45],
    "Public Score": [0.52, 0.45, 0.45, 0.40, 0.36, 0.38, 0.27, 0.42],
    "Strategy": ["Ensemble (3-Fold)", "Ensemble", "Baseline", "Huber+TTA", "MixUp+OneCycle", "Huber", "Baseline", "Transformer"],
    "Epochs": [30, 30, 30, 100, 160, 100, 70, 50]
}

df_res = pd.DataFrame(data)
df_res["Generalization Gap"] = df_res["Public Score"] - df_res["Private Score"]

## 2. Performance Comparison

### 2.1 Public vs. Private Leaderboard
Visualizing the stability of our models across the Public (seen test distribution) and Private (unseen) leaderboards.

In [ ]:
plt.figure(figsize=(12, 6))
df_melted = df_res.melt(id_vars=["Model", "Architecture"], value_vars=["Private Score", "Public Score"], var_name="Metric", value_name="R2 Score")
sns.barplot(data=df_melted, x="Model", y="R2 Score", hue="Metric", palette="viridis")
plt.title("Model Consistency: Public vs Private Scores", fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Analysis:**
- **ResNet18 Sub2** shows the highest scores and a small gap between Public and Private performance, indicating **excellent generalization**.
- **DenseNet121 v25** shows reasonable performance but lags behind ResNet. The larger gap suggests it may have partially overfit the public test set distribution or failed to capture features relevant to the private set.

## 3. Theoretical Deep Dive

### 3.1 Why did ResNet18 outperform DenseNet121? 
Conventional wisdom suggests deeper models (DenseNet) should outperform shallower ones (ResNet18). However, for this biomass task, the opposite occurred.

**Hypothesis 1: Feature Resolution & Texture**
- ResNet18 preserves spatial information relatively well in its earlier layers. Biomass estimation is a "texture" problem (density of leaves) rather than an "object detection" problem (finding a cat). The aggressive feature concatenation in DenseNet might have over-complicated the feature space for such a noisy regression task.

**Hypothesis 2: Overfitting & Dataset Size**
- Deep models like DenseNet121 have a higher capacity. Without massive datasets (ImageNet scale), they are prone to memorizing training noise. We attempted to mitigate this with **MixUp** (v21) and **TTA**, but the simpler inductive bias of ResNet18 proved more robust natively.

### 3.2 The Impact of Loss Functions
We experimented with **MSE (Mean Squared Error)** vs. **Huber Loss**.
- **Observation**: The target variables (e.g., `Dry_Dead_g`) had skewed distributions with some extreme values.
- **Theory**: MSE penalizes outliers quadratically ($L \propto e^2$), which can destabilize gradients. Huber Loss is linear ($L \propto |e|$) for large errors, making it robust to outliers.
- **Evidence**: Switching from MSE (v15) to Huber (v25) contributed to the stability of the DenseNet training.

### 3.3 MixUp and Scheduling (v21)
In **v21**, we deployed a complex strategy:
- **Dual Scheduler**: `OneCycleLR` (fast convergence) + `ReduceLROnPlateau` (fine-tuning). 
- **Result**: Performance (0.36 Public) was lower than v25 (0.40 Public). 
- **Correction**: This suggests that **MixUp might have been too strong a regularizer** for this regression task, "underfitting" the model or confusing the decision boundary between subtle biomass gradations.

## 4. Complexity vs. Performance
Does training longer yield better results? Let's check the correlation between Epochs and Score.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df_res, x="Epochs", y="Public Score", scatter_kws={'s':100}, line_kws={"color": "red"})
plt.title("Impact of Training Duration (Epochs) on Performance")
plt.ylabel("Public R2 Score")
plt.xlabel("Training Epochs")
plt.show()

**Observation**: Surprisingly, there is a **negative correlation**. The models trained for fewer epochs (ResNet18, ~30 epochs) performed better. 
- This reinforces the theory that the dataset is small enough that "state-of-the-art" long training schedules (like 160 epochs) lead to overfitting on the training noise rather than learning generalizable features.
- **Simple is Better**: A standard scheduler with fewer epochs was optimal.

## 5. References & Future Work

### References

1.  **ResNet**: He, K., et al. "Deep residual learning for image recognition." CVPR 2016. [Link](https://arxiv.org/abs/1512.03385)
2.  **DenseNet**: Huang, G., et al. "Densely connected convolutional networks." CVPR 2017. [Link](https://arxiv.org/abs/1608.06993)
3.  **MixUp**: Zhang, H., et al. "mixup: Beyond empirical risk minimization." ICLR 2018. [Link](https://arxiv.org/abs/1710.09412)
4.  **TTA**: Test-time augmentation involves averaging predictions from augmented copies of the test set.

### Future Recommendations
- **ResNet + Advanced Training**: Take the winning ResNet18 architecture but apply the **Huber Loss** (from DenseNet v25) to it. This combines the best architecture with the best loss function.
- **Higher Resolution**: The ResNet model utilized `Resize(256) -> CenterCrop(224)`. Moving to `300x300` (EfficientNet style) might expose more fine-grained texture details.